# task_05 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_05/data/")


In [2]:

users = pd.read_csv(base / "users.csv")
catalog = pd.read_csv(base / "catalog.csv")
subscriptions = pd.read_csv(base / "subscriptions.csv")
views = pd.read_csv(base / "views.csv")

users["signup_date"] = pd.to_datetime(users["signup_date"])
subscriptions["sub_start"] = pd.to_datetime(subscriptions["sub_start"])
subscriptions["sub_end"] = pd.to_datetime(subscriptions["sub_end"])
views["view_date"] = pd.to_datetime(views["view_date"])

Q2: Among users with at least 10 view events during their active subscription period, compute each user's average daily watch minutes considering only the calendar days on which they actually watched something. Which user_id has the lowest such average?

In [3]:
# Merge views with subscriptions to get active-period views
active_views = views.merge(subscriptions, on="user_id")
active_views = active_views[
    (active_views["view_date"] >= active_views["sub_start"]) &
    (active_views["view_date"] <= active_views["sub_end"])
].copy()

# Count view events per user
user_event_counts = active_views.groupby("user_id").size().rename("n_events")
eligible_users = user_event_counts[user_event_counts >= 10].index

# Filter to eligible users
eligible_views = active_views[active_views["user_id"].isin(eligible_users)].copy()

# Total watch minutes per user
total_min = eligible_views.groupby("user_id")["watch_min"].sum()

# Count distinct calendar days with at least one view per user
active_days = eligible_views.groupby("user_id")["view_date"].nunique()

# Average daily watch minutes on active-viewing days only
avg_daily = total_min / active_days

q2 = avg_daily.sort_values().index[0]
print(q2)

U08


Q5: In the final 7 active subscription days, and restricting to dates with at least 50 total watch minutes from those events, which date had the highest Documentary share of watch time?

In [4]:
views = pd.read_csv(base / "views.csv", parse_dates=["view_date"])
subscriptions = pd.read_csv(base / "subscriptions.csv", parse_dates=["sub_start", "sub_end"])
catalog = pd.read_csv(base / "catalog.csv")

merged = views.merge(subscriptions, on="user_id").merge(catalog[["title_id", "genre"]], on="title_id")
active = merged[(merged["view_date"] >= merged["sub_start"]) & (merged["view_date"] <= merged["sub_end"])].copy()
final7 = active[(active["sub_end"] - active["view_date"]).dt.days.between(0, 6)].copy()

daily = final7.groupby("view_date", as_index=False)["watch_min"].sum().rename(columns={"watch_min": "total_watch"})
doc = final7.loc[final7["genre"] == "Documentary"].groupby("view_date", as_index=False)["watch_min"].sum().rename(columns={"watch_min": "doc_watch"})
share = daily.merge(doc, on="view_date", how="left").fillna({"doc_watch": 0})
share = share[share["total_watch"] >= 50].copy()
share["doc_share"] = share["doc_watch"] / share["total_watch"]

q5 = share.sort_values(["doc_share", "view_date"], ascending=[False, True]).iloc[0]["view_date"].date().isoformat()
print(q5)

2024-03-28
